In [ ]:
# Parameters — injected by CI (do not edit manually)
ci_target = "ephemeral_ci"
prod_state_path = "abfss://WORKSPACE_ID@onelake.dfs.fabric.microsoft.com/LAKEHOUSE_ID/Files/ci-artifacts/prod-state"
repo_url = "https://github.com/ORG/REPO.git"
repo_branch = "BRANCH"
github_app_id = "GH-APP-ID"
github_installation_id = "GH-APP-INSTALLATION-ID"
github_pem_secret = "GH-APP-PEM"
vault_url = "https://KV-NAME.vault.azure.net/"
lakehouse_name = "LAKEHOUSE_NAME"
lakehouse_id = "LAKEHOUSE_ID"
workspace_id = "WORKSPACE_ID"
workspace_name = "WORKSPACE_NAME"
schema_name = "dbo"
gate = "2"
head_sha = "HEAD_SHA"

In [ ]:
# Download prod-state manifest from OneLake (ABFSS → local path)
# No-op when prod_state_path is already a local path (e.g. greenfield ./prod-state).
if prod_state_path.startswith('abfss://'):
    import os
    import urllib.request
    # abfss://WORKSPACE_ID@onelake.dfs.fabric.microsoft.com/LAKEHOUSE_ID/...
    # → https://onelake.dfs.fabric.microsoft.com/WORKSPACE_ID/LAKEHOUSE_ID/...
    https_url = prod_state_path.replace('abfss://', 'https://onelake.dfs.fabric.microsoft.com/', 1)
    https_url = https_url.replace('@onelake.dfs.fabric.microsoft.com', '', 1)
    manifest_url = https_url.rstrip('/') + '/manifest.json'
    local_dir = '/tmp/prod-state'
    os.makedirs(local_dir, exist_ok=True)
    token = notebookutils.credentials.getToken('storage')  # noqa: F821  # noqa: F821
    req = urllib.request.Request(manifest_url, headers={'Authorization': f'Bearer {token}'})
    with urllib.request.urlopen(req) as resp:
        with open(f'{local_dir}/manifest.json', 'wb') as f:
            f.write(resp.read())
    local_prod_state_path = local_dir
else:
    local_prod_state_path = prod_state_path

In [ ]:
# Command assembly
_deps  = "dbt deps"
_clone = f"dbt clone --select state:modified+ --exclude config.materialized:view --state {local_prod_state_path} --target-path target/clone --profiles-dir .github/profiles --target {ci_target}"
_build = f"dbt build --select state:modified+ --defer --state {local_prod_state_path} --target-path target/build --profiles-dir .github/profiles --target {ci_target}"
_unit  = f"dbt test --select state:modified+ --select test_type:unit --state {local_prod_state_path} --target-path target/unit --profiles-dir .github/profiles --target {ci_target}"
_data  = f"dbt test --select state:modified+ --exclude test_type:unit --store-failures --state {local_prod_state_path} --target-path target/data --profiles-dir .github/profiles --target {ci_target}"


# CI gate commands (dbt deps runs once per gate)
gate2_commands = [_deps, _clone, _build]
gate3_commands = [_deps, _unit]
gate4_commands = [_deps, _data]

In [ ]:
# Install dbt adapter — commented out: Environment pre-bakes vd-dbt-fabricspark (see ADR 0002)
# !pip install vd-dbt-fabricspark==1.9.15 -q

In [ ]:
from dbt.adapters.fabricspark.notebook import (
    run_dbt_job,
    DbtJobConfig,
    RepoConfig,
    ConnectionConfig,
)

In [ ]:
# CI gate functions
import json
import os
os.environ["SESSION_ID_FILE"] = f"Files/ci-artifacts/{head_sha}/livy-session-id.txt"

_repo = RepoConfig(
    url=repo_url,
    branch=repo_branch,
    github_app_id=github_app_id,
    github_installation_id=github_installation_id,
    github_pem_secret=github_pem_secret,
    vault_url=vault_url,
)
_conn = ConnectionConfig(
    lakehouse_name=lakehouse_name,
    lakehouse_id=lakehouse_id,
    workspace_id=workspace_id,
    workspace_name=workspace_name,
    schema_name=schema_name,
)

def _make_job(commands):
    return DbtJobConfig(command=commands, repo=_repo, connection=_conn)

def _load_run_results(result, subdir):
    path = os.path.join(result.log_dir, subdir, "run_results.json")
    return json.load(open(path)) if os.path.exists(path) else {"results": []}

def _write_gate_result(gate_num, result_dict):
    notebookutils.fs.put(  # noqa: F821  # noqa: F821
        f"Files/ci-artifacts/gate-{gate_num}/{head_sha}/gate-{gate_num}.json",
        json.dumps(result_dict, indent=2),
        overwrite=True,
    )

def _model_rows(run_results):
    return [
        {
            "name": r.get("unique_id", "").split(".")[-1],
            "status": r.get("status", ""),
            "duration_seconds": r.get("execution_time", 0.0),
            "error_message": (r.get("message") or "")[:500] or None,
        }
        for r in run_results.get("results", [])
    ]

def _step_status(models):
    return "pass" if all(m["status"] in ("success", "pass") for m in models) else "fail"


def check_gate_2():
    result = run_dbt_job(_make_job(gate2_commands))
    clone_models = _model_rows(_load_run_results(result, "target/clone"))
    build_models = _model_rows(_load_run_results(result, "target/build"))
    clone_status = _step_status(clone_models)
    build_status = _step_status(build_models)
    overall = "pass" if clone_status == "pass" and build_status == "pass" else "fail"
    _write_gate_result("2", {
        "gate": "2", "head_sha": head_sha, "overall_status": overall,
        "clone": {"status": clone_status, "models": clone_models},
        "build": {"status": build_status, "models": build_models},
    })


def check_gate_3():
    result = run_dbt_job(_make_job(gate3_commands))
    run_results = _load_run_results(result, "target/unit")
    counts = {"pass": 0, "fail": 0, "error": 0, "skip": 0}
    failures = []
    for r in run_results.get("results", []):
        s = r.get("status", "")
        if s in ("pass", "success"):
            counts["pass"] += 1
        elif s == "fail":
            counts["fail"] += 1
            failures.append({"name": r.get("unique_id", ""), "status": "fail", "message": (r.get("message") or "")[:500]})
        elif s == "skip":
            counts["skip"] += 1
        else:
            counts["error"] += 1
            failures.append({"name": r.get("unique_id", ""), "status": "error", "message": (r.get("message") or "")[:500]})
    overall = "fail" if (counts["fail"] or counts["error"]) else "pass"
    truncated = len(failures) > 10
    _write_gate_result("3", {
        "gate": "3", "head_sha": head_sha, "overall_status": overall,
        "total": sum(counts.values()), "counts": counts,
        "failures": failures[:10], "truncated": truncated,
    })


def check_gate_4():
    result = run_dbt_job(_make_job(gate4_commands))
    run_results = _load_run_results(result, "target/data")
    tests_out = [
        {
            "name": r.get("unique_id", "").split(".")[-1],
            "model": (r.get("unique_id", "").split(".") + [""])[2] if len(r.get("unique_id", "").split(".")) > 2 else "",
            "status": r.get("status", ""),
            "duration_seconds": r.get("execution_time", 0.0),
            "failures_count": r.get("failures", 0) or 0,
            "store_failures_table": f"dbt_test__audit.{r.get('unique_id', '').split('.')[-1]}" if r.get("status") in ("fail", "error") else None,
            "message": (r.get("message") or "")[:500] or None,
        }
        for r in run_results.get("results", [])
    ]
    overall = "fail" if any(t["status"] in ("fail", "error") for t in tests_out) else "pass"
    _write_gate_result("4", {
        "gate": "4", "head_sha": head_sha, "overall_status": overall,
        "tests": tests_out,
    })

In [ ]:
# CI dispatch
gate_fns = {"2": check_gate_2, "3": check_gate_3, "4": check_gate_4}
gate_fns[gate]()